In [ ]:
import cv2
import numpy as np
import torch
import torchvision

from boxmot import create_tracker
from boxmot.structures import Boxes, Detections, Frame, MaskBatch
from boxmot.trackers import TrackerSpec

# Load a pre-trained Mask R-CNN model from torchvision
device = torch.device('cpu')  # Change to 'cuda' if you have a GPU available
segmentation_model = torchvision.models.detection.maskrcnn_resnet50_fpn(pretrained=True)
segmentation_model.eval().to(device)

tracker = create_tracker(TrackerSpec(name='sam2mot', geometry='aabb'))

# Open the video file
vid = cv2.VideoCapture(0)
frame_index = 0

# Function to generate a unique color for each track ID
def get_color(track_id):
    np.random.seed(int(track_id))
    return tuple(np.random.randint(0, 255, 3).tolist())

while True:
    ret, im = vid.read()
    if not ret:
        break

    # Convert frame to tensor and move to device
    frame_tensor = torchvision.transforms.functional.to_tensor(im).unsqueeze(0).to(device)

    # Run the Mask R-CNN model to detect bounding boxes and masks
    with torch.no_grad():
        results = segmentation_model(frame_tensor)[0]

    # Extract detections (bounding boxes, masks, and scores)
    dets = []
    masks = []
    confidence_threshold = 0.5

    for i, score in enumerate(results['scores']):
        if score >= confidence_threshold:
            # Extract bounding box and score
            x1, y1, x2, y2 = results['boxes'][i].cpu().numpy()
            conf = score.item()
            cls = results['labels'][i].item()  # Assuming 'labels' represents the class
            dets.append([x1, y1, x2, y2, conf, cls])

            # Extract mask and add to list
            mask = results['masks'][i, 0].cpu().numpy()  # Use the first channel (binary mask)
            masks.append(mask)

    # Convert once at the detector boundary into canonical CPU structures
    rows = torch.tensor(dets, dtype=torch.float32) if dets else torch.empty((0, 6), dtype=torch.float32)
    if masks:
        mask_values = torch.from_numpy(np.stack(masks)).gt(0.5).contiguous()
        foreground = mask_values.flatten(1).any(1)
        rows, mask_values = rows[foreground].contiguous(), mask_values[foreground].contiguous()
    else:
        mask_values = torch.empty((0, im.shape[0], im.shape[1]), dtype=torch.bool)
    rgb = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
    frame = Frame(
        image=torch.from_numpy(rgb).permute(2, 0, 1).contiguous(),
        sample_id=f'camera-0:{frame_index:012d}', sequence_id='camera-0', frame_index=frame_index,
    )
    detections = Detections(
        geometry=Boxes(rows[:, :4].contiguous()), scores=rows[:, 4].contiguous(),
        class_ids=rows[:, 5].to(dtype=torch.int64).contiguous(), sample_id=frame.sample_id,
        masks=MaskBatch(mask_values),
    )
    tracks = tracker.update(detections, frame=frame)
    frame_index += 1

    # Draw segmentation masks and bounding boxes in a single loop
    if len(tracks) > 0:
        track_masks = tracks.masks.values if tracks.masks is not None else None

        # Iterate over canonical, track-row-aligned geometry and masks
        for index, (box, track_id, conf, cls) in enumerate(zip(
            tracks.geometry.values.round().to(torch.int64).tolist(), tracks.track_ids.tolist(),
            tracks.scores.tolist(), tracks.class_ids.tolist(),
        )):
            color = get_color(track_id)  # Use unique color for each track

            # Draw the segmentation mask on the image
            if track_masks is not None:
                mask = track_masks[index].numpy()
                # Blend mask color with the image
                im[mask] = im[mask] * 0.5 + np.array(color) * 0.5

            # Draw the bounding box
            x1, y1, x2, y2 = box
            cv2.rectangle(im, (x1, y1), (x2, y2), color, 2)

            # Add text with ID, confidence, and class
            cv2.putText(im, f'ID: {track_id}, Conf: {conf:.2f}, Class: {cls}',
                        (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    # Display the image
    cv2.imshow('Segmentation Tracking', im)

    # Break on pressing q or space
    key = cv2.waitKey(1) & 0xFF
    if key == ord(' ') or key == ord('q'):
        break

vid.release()
cv2.destroyAllWindows()